# MAGPIE Test — Point Cloud Filtering

Test Open3D statistical and radius outlier removal on the segmented point cloud.

Run Cell 1 (ROS loader) → Cell 2 (connect) → then the test cells.

In [ ]:
# ROS pre-loader — makes rclpy importable in VS Code
import sys, os, ctypes, glob

for _d in [
    '/opt/ros/humble/lib/x86_64-linux-gnu',
    '/opt/ros/humble/lib',
    '/home/user/ws_ctrl/install/magpie_msgs/lib',
    '/home/user/ws_ctrl/install/magpie_control/lib',
]:
    for _so in sorted(glob.glob(_d + '/*.so*')):
        try: ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except: pass

for _p in [
    '/opt/ros/humble/local/lib/python3.10/dist-packages',
    '/opt/ros/humble/lib/python3.10/site-packages',
    '/home/user/ws_ctrl/install/magpie_msgs/local/lib/python3.10/dist-packages',
    '/home/user/ws_ctrl/install/magpie_control/lib/python3.10/site-packages',
    '/home/user/.local/lib/python3.10/site-packages',
    '/home/user/magpie_control/src',
    '/home/user/magpie_control/scripts',
]:
    if _p not in sys.path: sys.path.insert(0, _p)

import rclpy
print('rclpy OK')

In [ ]:
import time, json, tempfile, base64, socket as _sock
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import cv2, open3d as o3d
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi']     = 80

from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy
from sensor_msgs.msg import Image as RosImage, CameraInfo
from geometry_msgs.msg import PoseStamped
from cv_bridge import CvBridge
from magpie_control import poses
from magpie_control.homog_utils import homog_xform, R_krot
from pointcloud_utils import build_segmented_pcd

SAM3_SOCK   = '/tmp/sam3.sock'
OBJECT      = 'red block'
_TCP_TO_CAM = homog_xform(R_krot([0, 0, 1], -np.pi/2), [0, 0, 0.120])

def _quat_to_rv(w, x, y, z):
    if w < 0.0: w, x, y, z = -w, -x, -y, -z   # q and -q are one rotation -> force canonical (<=pi) rotvec
    a = 2.0 * np.arccos(np.clip(w, -1, 1))
    s = np.sin(a / 2)
    return np.zeros(3) if s < 1e-10 else a * np.array([x, y, z]) / s

img_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.TRANSIENT_LOCAL)
inf_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.VOLATILE)

class Snap(Node):
    def __init__(self):
        super().__init__('magpie_test')
        self.bridge = CvBridge()
        self.color = self.depth = self.caminfo = self.tcp = None
        NS = '/camera/gripper_camera/camera'
        self.create_subscription(RosImage,   NS+'/color/image_raw',
            lambda m: setattr(self,'color', self.bridge.imgmsg_to_cv2(m,'rgb8')), img_qos)
        self.create_subscription(RosImage,   NS+'/depth/image_rect_raw',
            lambda m: setattr(self,'depth', self.bridge.imgmsg_to_cv2(m,'passthrough')), img_qos)
        self.create_subscription(CameraInfo, NS+'/color/camera_info',
            lambda m: setattr(self,'caminfo', m), inf_qos)
        self.create_subscription(PoseStamped,'/arm/tcp_pose', self._tcp_cb, 1)

    def _tcp_cb(self, m):
        p = m.pose
        rv = _quat_to_rv(p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z)
        self.tcp = poses.pose_vec_to_mtrx([p.position.x, p.position.y, p.position.z, *rv])

    def wait(self, to=20.):
        t0 = time.time()
        while time.time()-t0 < to:
            rclpy.spin_once(self, timeout_sec=0.15)
            if all(v is not None for v in [self.color, self.depth, self.caminfo, self.tcp]):
                return True
        return False

    def spin(self, n=8):
        for _ in range(n): rclpy.spin_once(self, timeout_sec=0.15)

try: rclpy.init()
except RuntimeError: pass
try: node.destroy_node()
except: pass
node = Snap()
ok = node.wait(20.)
print('Ready:', ok, '| color:', node.color is not None,
      '| depth:', node.depth is not None, '| tcp:', node.tcp is not None)

In [ ]:
# SAM3 query helper
def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp = tempfile.mktemp(suffix='.jpg')
    cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
            s.connect(sock_path)
            s.sendall((json.dumps({'image': tmp, 'query': query}) + '\n').encode())
            raw = b''
            while True:
                chunk = s.recv(65536)
                if not chunk: break
                raw += chunk
        d = json.loads(raw.decode().strip())
        if 'error' in d: raise RuntimeError(d['error'])
        boxes  = np.array(d['boxes'],  dtype=float)
        scores = np.array(d['scores'], dtype=float)
        mask   = None
        if d.get('mask_b64') and len(boxes) > 0:
            raw2 = base64.b64decode(d['mask_b64'])
            h, w = d['mask_shape']
            mask = np.frombuffer(raw2, dtype=np.uint8).reshape(h, w).astype(bool)
        return boxes, scores, mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

# Snapshot + SAM3
node.spin()
color   = node.color.copy()
depth   = node.depth.copy()
tcp     = node.tcp.copy()
caminfo = node.caminfo

boxes, scores, mask = sam3_query(color, OBJECT)
best = int(np.argmax(scores)) if len(scores) > 0 else 0
print(f'SAM3  score={scores[best]:.3f}  mask_px={mask.sum() if mask is not None else 0}')

# Show detection
vis = color.copy()
if mask is not None:
    vis[mask] = (vis[mask]*.4 + np.array([0,200,0])*.6).astype(np.uint8)
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(vis)
x1,y1,x2,y2 = boxes[best].astype(int)
ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=3,ec='lime',fc='none'))
ax.set_title(f'SAM3 mask — "{OBJECT}"'); ax.axis('off')
plt.tight_layout(); plt.show()

---
## Open3D Outlier Removal Comparison

Compare **statistical** vs **radius** outlier removal at different settings.

In [ ]:
# Build raw point cloud from SAM3 mask (no denoising)
pts_raw, pcd_raw = build_segmented_pcd(mask, depth, caminfo.k, tcp, _TCP_TO_CAM)
print(f'Raw points: {len(pts_raw)}')

# --- Statistical outlier removal at different std_ratio values ---
stat_configs = [
    ('std=2.0 (default)', 20, 2.0),
    ('std=1.0',           20, 1.0),
    ('std=0.5 (tight)',   30, 0.5),
]

def stat_filter(pcd, nb_neighbors, std_ratio):
    cleaned, _ = pcd.remove_statistical_outlier(
        nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    return np.asarray(cleaned.points)

fig = plt.figure(figsize=(18, 5))
configs = [('Raw', pts_raw)] + [(lbl, stat_filter(pcd_raw, nb, sr))
                                  for lbl, nb, sr in stat_configs]
for i, (lbl, pts) in enumerate(configs):
    ax = fig.add_subplot(1, 4, i+1, projection='3d')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=pts[:,2], cmap='plasma', s=3, alpha=.6)
    ax.set_title(f'{lbl}\n{len(pts)} pts', fontsize=9)
    ax.set_xlabel('X',fontsize=7); ax.set_ylabel('Y',fontsize=7); ax.set_zlabel('Z',fontsize=7)
    ax.tick_params(labelsize=6)
plt.suptitle('Statistical Outlier Removal'); plt.tight_layout(); plt.show()

In [ ]:
# --- Radius outlier removal ---
# Keeps only points that have >= min_points neighbours within radius
radius_configs = [
    ('r=0.01 n=5',  0.010, 5),
    ('r=0.005 n=5', 0.005, 5),
    ('r=0.005 n=10',0.005, 10),
]

def radius_filter(pcd, radius, min_points):
    cleaned, _ = pcd.remove_radius_outlier(nb_points=min_points, radius=radius)
    return np.asarray(cleaned.points)

fig = plt.figure(figsize=(18, 5))
configs2 = [('Raw', pts_raw)] + [(lbl, radius_filter(pcd_raw, r, n))
                                    for lbl, r, n in radius_configs]
for i, (lbl, pts) in enumerate(configs2):
    ax = fig.add_subplot(1, 4, i+1, projection='3d')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=pts[:,2], cmap='plasma', s=3, alpha=.6)
    ax.set_title(f'{lbl}\n{len(pts)} pts', fontsize=9)
    ax.set_xlabel('X',fontsize=7); ax.set_ylabel('Y',fontsize=7); ax.set_zlabel('Z',fontsize=7)
    ax.tick_params(labelsize=6)
plt.suptitle('Radius Outlier Removal'); plt.tight_layout(); plt.show()

In [ ]:
# --- Side-by-side top-down view (easier to judge than 3D) ---
best_stat   = stat_filter(pcd_raw, 30, 0.5)
best_radius = radius_filter(pcd_raw, 0.005, 5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (lbl, pts) in zip(axes, [
    ('Raw',                         pts_raw),
    ('Statistical\nnb=30 std=0.5',  best_stat),
    ('Radius\nr=0.005 n=5',         best_radius),
]):
    ax.scatter(pts[:,0], pts[:,1], c=pts[:,2], cmap='plasma', s=4, alpha=.6)
    ax.set_title(f'{lbl}\n{len(pts)} pts')
    ax.set_xlabel('X m'); ax.set_ylabel('Y m')
    ax.set_aspect('equal')
plt.suptitle('Top-down comparison'); plt.tight_layout(); plt.show()

print('Recommendation: pick the config where the block is a tight rectangle')
print(f'  Raw          : {len(pts_raw):5d} pts')
print(f'  Stat nb=30 std=0.5 : {len(best_stat):5d} pts')
print(f'  Radius r=0.005 n=5 : {len(best_radius):5d} pts')

---
## RANSAC Plane Removal + DBSCAN

Statistical/radius removal doesn't work because the table has consistent neighbors too.
RANSAC finds the dominant plane (table), removes it, DBSCAN clusters what's left (the block).

In [ ]:
# RANSAC — detect and remove the dominant plane (table)
plane_model, inliers = pcd_raw.segment_plane(
    distance_threshold=0.005,   # 5mm tolerance for table plane
    ransac_n=3,
    num_iterations=1000)
[a, b, c, d] = plane_model
print(f'Table plane: {a:.3f}x + {b:.3f}y + {c:.3f}z + {d:.3f} = 0')

object_cloud = pcd_raw.select_by_index(inliers, invert=True)
pts_above    = np.asarray(object_cloud.points)
print(f'Table pts: {len(inliers)}  |  Above-plane pts: {len(pts_above)}')

# DBSCAN — cluster remaining points, keep largest cluster (the block)
labels = np.array(object_cloud.cluster_dbscan(eps=0.008, min_points=10, print_progress=False))
if labels.max() < 0:
    print('No clusters found — try increasing eps')
    pts_block = pts_above
else:
    counts       = np.bincount(labels[labels >= 0])
    best_cluster = int(np.argmax(counts))
    pts_block    = pts_above[labels == best_cluster]
    print(f'Clusters: {labels.max()+1}  |  Block cluster: {len(pts_block)} pts')

# Top-down comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (lbl, pts) in zip(axes, [
    ('Raw\n(all)',             pts_raw),
    ('Above table\n(RANSAC)', pts_above),
    ('Block only\n(DBSCAN)',  pts_block),
]):
    ax.scatter(pts[:,0], pts[:,1], c=pts[:,2], cmap='plasma', s=5, alpha=.7)
    ax.set_title(f'{lbl}\n{len(pts)} pts')
    ax.set_xlabel('X m'); ax.set_ylabel('Y m')
    ax.set_aspect('equal')
plt.suptitle('RANSAC + DBSCAN'); plt.tight_layout(); plt.show()

---
## Apply Best Filter + PCA

Once you pick the best config above, run this cell to confirm PCA looks correct.

In [ ]:
from pointcloud_utils import analyse_pcd

# Statistical outlier removal — removes gripper edge noise
cleaned, _ = pcd_raw.remove_statistical_outlier(nb_neighbors=30, std_ratio=0.5)
pts_clean  = np.asarray(cleaned.points)
print(f'Raw: {len(pts_raw)} pts  →  Clean: {len(pts_clean)} pts')

# PCA on cleaned block surface
pca = analyse_pcd(pts_clean)
cen = pca['centroid']; ext = pca['extent_m']; ang = pca['grasp_angle_deg']
print(f'Extent  major={ext[0]*1e3:.1f}mm  minor={ext[1]*1e3:.1f}mm  normal={ext[2]*1e3:.1f}mm')
print(f'Grasp angle : {ang:.1f} deg')
print(f'Auto offset : {np.clip(ext[2]/4, 0.005, 0.015)*1e3:.0f} mm')

ptc = pts_clean - cen
ev, evec = np.linalg.eigh(np.cov(ptc.T))
idx = np.argsort(ev)[::-1]; evec = evec[:,idx]; sc = np.sqrt(np.abs(ev[idx]))
t = np.radians(ang); L = sc[0]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Raw
axes[0].scatter(pts_raw[:,0], pts_raw[:,1], c=pts_raw[:,2], cmap='plasma', s=4, alpha=.5)
axes[0].set_title(f'Raw\n{len(pts_raw)} pts'); axes[0].set_aspect('equal')
axes[0].set_xlabel('X m'); axes[0].set_ylabel('Y m')

# Clean
axes[1].scatter(pts_clean[:,0], pts_clean[:,1], c=pts_clean[:,2], cmap='plasma', s=4, alpha=.7)
axes[1].set_title(f'Statistical nb=30 std=0.5\n{len(pts_clean)} pts'); axes[1].set_aspect('equal')
axes[1].set_xlabel('X m'); axes[1].set_ylabel('Y m')

# PCA top-down
axes[2].scatter(pts_clean[:,0], pts_clean[:,1], c=pts_clean[:,2], cmap='plasma', s=6, alpha=.8)
axes[2].annotate('', xy=(cen[0]+L*np.cos(t), cen[1]+L*np.sin(t)),
                 xytext=(cen[0]-L*np.cos(t), cen[1]-L*np.sin(t)),
                 arrowprops=dict(arrowstyle='<->', color='red', lw=2.5))
axes[2].plot(*cen[:2], 'y+', ms=14, mew=2)
axes[2].set_aspect('equal'); axes[2].set_xlabel('X m'); axes[2].set_ylabel('Y m')
axes[2].set_title(f'PCA  angle={ang:.1f}deg  major={ext[0]*1e3:.0f}mm')
plt.suptitle('Statistical + PCA'); plt.tight_layout(); plt.show()

In [ ]:
# Z-height histogram — shows if block and table are at different heights
z = pts_raw[:, 2]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(z, bins=100, color='steelblue', edgecolor='none')
ax.set_xlabel('Z (m)'); ax.set_ylabel('point count')
ax.set_title('Z-height distribution of raw point cloud')
plt.tight_layout(); plt.show()
print(f'Z range: {z.min():.4f} – {z.max():.4f} m')
print(f'Median: {np.median(z):.4f} m')

In [ ]:
import importlib, pointcloud_utils; importlib.reload(pointcloud_utils)
from pointcloud_utils import denoise_pcd, analyse_pcd, top_layer

# Denoise keeps all valid points (for position)
pcd_cln, pts_cln = denoise_pcd(pcd_raw)

# Apply Z-layer filter only for angle computation
pts_top = top_layer(pts_cln)
print(f'Raw: {len(pts_raw)} pts  →  Denoised: {len(pts_cln)} pts  →  Top layer: {len(pts_top)} pts')
print(f'Top layer Z range: {pts_top[:,2].min()*1000:.0f}mm – {pts_top[:,2].max()*1000:.0f}mm')

pca_z = analyse_pcd(pts_top)
ang_z = pca_z['grasp_angle_deg']
ext_z = pca_z['extent_m']
cen_z = pca_z['centroid']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(pts_top[:,0], pts_top[:,1], c=pts_top[:,2], cmap='plasma', s=5, alpha=.8)
axes[0].set_title(f'Top layer only (angle)\n{len(pts_top)} pts')
axes[0].set_aspect('equal'); axes[0].set_xlabel('X m'); axes[0].set_ylabel('Y m')

t = np.radians(ang_z)
ptc_z = pts_top - cen_z
ev_z, evec_z = np.linalg.eigh(np.cov(ptc_z.T))
sc_z = np.sqrt(np.abs(ev_z[np.argsort(ev_z)[::-1]]))
L_z = sc_z[0]
axes[1].scatter(pts_top[:,0], pts_top[:,1], c=pts_top[:,2], cmap='plasma', s=6, alpha=.8)
axes[1].annotate('', xy=(cen_z[0]+L_z*np.cos(t), cen_z[1]+L_z*np.sin(t)),
                 xytext=(cen_z[0]-L_z*np.cos(t), cen_z[1]-L_z*np.sin(t)),
                 arrowprops=dict(arrowstyle='<->', color='red', lw=2.5))
axes[1].plot(*cen_z[:2], 'y+', ms=14, mew=2)
axes[1].set_aspect('equal'); axes[1].set_xlabel('X m'); axes[1].set_ylabel('Y m')
axes[1].set_title(f'PCA  angle={ang_z:.1f}deg  major={ext_z[0]*1e3:.0f}mm')
plt.suptitle('Z-layer Filter for Angle Only'); plt.tight_layout(); plt.show()